In [1]:
from pathlib import Path

import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.metrics import accuracy_score, precision_score, f1_score, confusion_matrix, recall_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import FeatureUnion
from sklearn.preprocessing import FunctionTransformer
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import torch.optim as optim
from sklearn.model_selection import train_test_split

from sklearn.metrics import accuracy_score, classification_report
import noisereduce as nr

from features.extract_features import get_mel_spectrogram, get_mfcc
from models import DecisionTreeModel, LSTMModel, GRUModel
from utils import min_max_scaler, pad_audio
from denoise.denoise_methods import noise_reduce_denoise
from scipy.signal import butter, sosfilt

from torchmetrics import Accuracy, Precision, Recall, F1Score, ConfusionMatrix

c:\Users\josel\OneDrive\Desktop\NCIA\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print(torch.cuda.is_available())

True


In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

DCASE2024_ROOT_PATH = Path.cwd().parent / "data/raw/dcase-2024"

DCASE2024_TRAIN_PATH = DCASE2024_ROOT_PATH / "Train"
DCASE2024_DEV_PATH = DCASE2024_ROOT_PATH / "Dev"
DCASE2024_EVAL_PATH = DCASE2024_ROOT_PATH / "Eval"

DCASE2022_ROOT_PATH = Path.cwd().parent / "data/raw/dcase-2022"

DCASE2022_TRAIN_PATH = DCASE2022_ROOT_PATH / "Train"
DCASE2022_DEV_PATH = DCASE2022_ROOT_PATH / "Dev"

# ===============================================================
# ===============================================================

SAMPLE_RATE = 16_000

MEL_SPECTROGRAM_PARAMS = {
    "n_fft": 1024,
    "hop_length": 512,
    "n_mels": 128,
    "window": "hann",
    "center": True,
    "pad_mode": "reflect",
    "power": 2.0,
}

MFCC_PARAMS = {
    "n_mfcc": 20,
    "n_fft": 143,
    "hop_length": 512,
    "n_mels": 128,
    "dct_type": 2,
    "norm": "ortho",
    "lifter": 0,
}

In [4]:
def load_audio(audio):
    signal, sample_rate = librosa.load(audio, mono=True, sr=SAMPLE_RATE)

    return signal, sample_rate

def pad_or_trim(signal, target_len):
    if len(signal) > target_len:
        return signal[:target_len]
    elif len(signal) < target_len:
        return np.pad(signal, (0, target_len - len(signal)), 'constant')
   
    return signal

def rms_normalize(signal, target_rms=0.1, eps=1e-8):
    rms = np.sqrt(np.mean(signal**2) + eps)
    gain = target_rms / (rms + eps)

    return (signal * gain).astype(np.float32)

def peak_normalize(signal):
    peak = np.max(np.abs(signal))
    if peak > 0:
        return signal / peak
    return signal


def preprocess_data(signals, padding_or_trim=True, denoise_method=None, normalize_method=None):
    mel_features = []
    mfcc_features = []
    target_len = 10 * SAMPLE_RATE

    for signal in signals:
        if padding_or_trim:
            signal = pad_or_trim(signal, target_len)
        if denoise_method:
            signal = denoise_method(signal)
        if normalize_method:
            signal = normalize_method(signal)
        
        mel = get_mel_spectrogram(signal, MEL_SPECTROGRAM_PARAMS)
        mfcc = get_mfcc(signal, MFCC_PARAMS)

        # norm_mel = min_max_scaler(mel)
        # norm_mfcc = min_max_scaler(mfcc) 

        mel_features.append(mel)
        mfcc_features.append(mfcc)

    return mel_features, mfcc_features

def bandpass_filter(signal, sr=16000, low=80, high=7500, order=4):
    nyq = 0.5 * sr
    low_n = low / nyq
    high_n = high / nyq
    sos = butter(order, [low_n, high_n], btype="bandpass", output="sos")
    return sosfilt(sos, signal).astype(np.float32)

def spectral_gate(signal, sr=16000, noise_sec=0.25):
    n = int(noise_sec * sr)
    noise_clip = signal[:n]
    return nr.reduce_noise(y=signal, y_noise=noise_clip, sr=sr).astype(np.float32)

def highpass_filter(signal, sr=16000, cutoff=80, order=4):
    nyq = 0.5 * sr
    cutoff_n = cutoff / nyq
    sos = butter(order, cutoff_n, btype="highpass", output="sos")
    return sosfilt(sos, signal).astype(np.float32)

# Leitura dos áudios


In [5]:
def read_dcase_dev_set(path, ignore_machine_types=None, only_machine_type=None):
    if ignore_machine_types is None:
        ignore_machine_types = []

    X_train, y_train, mtype_train = [], [], []
    X_test, y_test, mtype_test = [], [], []

    for machine_dir in Path(path).iterdir():
        if not machine_dir.is_dir():
            continue

        mtype = machine_dir.name
        if mtype in ignore_machine_types:
            continue

        if only_machine_type is not None and mtype != only_machine_type:
            continue

        for section in ["train", "test"]:
            data_dir = machine_dir / section
            if not data_dir.exists():
                continue

            wavs = list(data_dir.glob("*.wav"))
            for audio_file in tqdm(wavs, desc=f"{mtype} - {section}"):
                try:
                    signal, _ = load_audio(str(audio_file))

                    if section == "train":
                        label = 0
                        X_train.append(signal)
                        y_train.append(label)
                        mtype_train.append(mtype)

                    else:
                        label = 1 if "anomaly" in audio_file.name.lower() else 0
                        X_test.append(signal)
                        y_test.append(label)
                        mtype_test.append(mtype)

                except Exception as e:
                    print(f"Erro ao processar {audio_file.name}: {e}")

    return X_train, X_test, y_train, y_test, mtype_train, mtype_test


def read_dcase_train_set(path):
    X = []
    machine_type = []
    y = []

    for machine_dir in Path(path).iterdir():
        if not machine_dir.is_dir():
            continue

        data_dir = machine_dir / "train"
        if not data_dir.exists():
            continue

        for audio_file in tqdm(data_dir.iterdir(), desc="Lendo train"):
            if audio_file.suffix != ".wav":
                continue
            try:
                label = 0

                signal, _ = load_audio(audio_file)

                X.append(signal)
                machine_type.append(machine_dir.name)
                y.append(label)

            except Exception as e:
                print(f"Erro ao processar {audio_file.name}: {e}")

    return X, machine_type, y


def read_dcase_eval_set(path):
    X = []
    machine_type = []
    y = []

    for machine_dir in Path(path).iterdir():
        if not machine_dir.is_dir():
            continue

        data_dir = machine_dir / "test"
        if not data_dir.exists():
            continue

        for audio_file in tqdm(data_dir.iterdir(), desc="Lendo eval"):
            if audio_file.suffix != ".wav":
                continue
            try:
                if "anomaly" in audio_file.name:
                    label = 1
                else:
                    label = 0

                signal, _ = load_audio(audio_file)

                X.append(signal)
                machine_type.append(machine_dir.name)
                y.append(label)

            except Exception as e:
                print(f"Erro ao processar {audio_file.name}: {e}")

    return X, machine_type, y

In [6]:
# Conjuntos que o artigo usa

# 25.199 áudios (treino e teste) - DCASE 2022/MIMII DG
#X_dcase22, mtypes_dcase22, y_dcase22 = read_dcase_dev_set(DCASE2022_DEV_PATH)

# # 9.000 áudios (treino) - DCASE 2024
# X_dcase24_train, mtypes_dcase24_train, y_dcase24_train = read_dcase_train_set(DCASE2024_TRAIN_PATH)

# # 1.800 áudios (teste) - DCASE 2024
# X_dcase24_test, mtypes_dcase24_test, y_dcase24_test = read_dcase_eval_set(DCASE2024_EVAL_PATH)

# # União do conjunto DCASE 2024
# X_dcase24 = X_dcase24_test + X_dcase24_train
# mtypes_dcase24 = mtypes_dcase24_test + mtypes_dcase24_train
# y_dcase24 = y_dcase24_test + y_dcase24_train

In [7]:
signals_train, signals_test, labels_train, labels_test, mtype_train, mtype_test = read_dcase_dev_set(DCASE2022_DEV_PATH, only_machine_type='fan')

print()
print('=' * 45)
print('Leitura Concluída!')
print(f'Total de amostras: {len(signals_train) + len(signals_test)}')
print(f'Amostras de treino: {len(signals_train)}')
print(f'Amostras de testes: {len(signals_test)}')
print(f'Máquinas de treino: {np.unique(mtype_train)}')
print(f'Máquinas de teste: {np.unique(mtype_test)}')
print('=' * 45)
print()

fan - train:   0%|          | 0/3000 [00:00<?, ?it/s]

fan - test: 100%|██████████| 600/600 [00:01<00:00, 498.67it/s]


Leitura Concluída!
Total de amostras: 3600
Amostras de treino: 3000
Amostras de testes: 600
Máquinas de treino: ['fan']
Máquinas de teste: ['fan']



In [8]:
# i = 1000
# mel, mfcc, label, mtype = test_dataset[i]

# print(f"Formato de um Mel-Spectrograma: {mel.shape}")
# print(f"Formato de um MFCC: {mfcc.shape}")

# print()
# print()

# plt.figure(figsize=(12, 5))
# librosa.display.specshow(mel.numpy(), sr=24000, hop_length=512, x_axis="time", y_axis="mel")

# plt.colorbar(format="%+2.0f dB")
# plt.title(f"Visualização do Mel-Spectrograma (128x469) - '{mtype}' {'Normal' if label else 'Anomalia'}")
# plt.tight_layout()
# plt.show()

In [9]:
class MachineAudioDataset(Dataset):
    def __init__(self, features, labels):
       self.features = features
       self.labels = labels

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        feature = self.features[idx]
        label = self.labels[idx]

        return (torch.tensor(feature, dtype=torch.float32),
                torch.tensor(label, dtype=torch.long))

In [10]:
def train(model, optimizer, criterion, train_loader, epochs):
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        total_n = 0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device, dtype=torch.float32)
            y_batch = y_batch.to(device, dtype=torch.long)

            optimizer.zero_grad()

            logits = model(X_batch)
            loss = criterion(logits, y_batch)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            bs = X_batch.size(0)
            total_loss += loss.item() * bs
            total_n += bs

        mean_loss = total_loss / max(1, total_n)
        print(f"Epoch {epoch + 1}/{epochs}, Loss: {mean_loss:.4f}")

@torch.no_grad()
def evaluate(model, data_loader, num_classes=2):
    model.eval()

    acc_metric = Accuracy(task="multiclass", num_classes=num_classes).to(device)
    prec_metric = Precision(task="multiclass", num_classes=num_classes, average="macro").to(device)
    rec_metric = Recall(task="multiclass", num_classes=num_classes, average="macro").to(device)
    f1_metric = F1Score(task="multiclass", num_classes=num_classes, average="macro").to(device)
    cm_metric = ConfusionMatrix(task="multiclass", num_classes=num_classes).to(device)

    for X_batch, y_batch in data_loader:
        X_batch = X_batch.to(device, dtype=torch.float32)  # (B, T, C)
        y_batch = y_batch.to(device, dtype=torch.long)

        logits = model(X_batch)

        acc_metric.update(logits, y_batch)
        prec_metric.update(logits, y_batch)
        rec_metric.update(logits, y_batch)
        f1_metric.update(logits, y_batch)
        cm_metric.update(logits, y_batch)

    acc = acc_metric.compute().item()
    prec = prec_metric.compute().item()
    rec = rec_metric.compute().item()
    f1 = f1_metric.compute().item()
    cm = cm_metric.compute().detach().cpu().numpy()

    acc_metric.reset(); prec_metric.reset(); rec_metric.reset(); f1_metric.reset(); cm_metric.reset()
    return acc, prec, rec, f1, cm

In [11]:
X_all = signals_train + signals_test
y_all = labels_train + labels_test

X_train, X_test, y_train, y_test = train_test_split(
    X_all,
    y_all,
    test_size=0.2,
    random_state=42,
    shuffle=True,
    #stratify=y_all
)

In [12]:
def to_time_major(feat_ct: np.ndarray) -> np.ndarray:
    return feat_ct.T.astype(np.float32)

def fit_zscore(train_feats_tc: list[np.ndarray]) -> tuple[np.ndarray, np.ndarray]:
    all_frames = np.concatenate(train_feats_tc, axis=0)  # (sum_T, C)
    mean = all_frames.mean(axis=0).astype(np.float32)
    std = (all_frames.std(axis=0) + 1e-8).astype(np.float32)
    return mean, std

def apply_zscore(feats_tc: list[np.ndarray], mean: np.ndarray, std: np.ndarray) -> list[np.ndarray]:
    out = []
    for x in feats_tc:
        out.append(((x - mean) / std).astype(np.float32))
    return out

In [13]:
#denoise_method = lambda x: bandpass_filter(x, SAMPLE_RATE, low=80, high=7500)
denoise_method = None

train_mel, train_mfcc = preprocess_data(X_train, denoise_method=denoise_method, normalize_method=rms_normalize)
test_mel,  test_mfcc  = preprocess_data(X_test,  denoise_method=denoise_method, normalize_method=rms_normalize)

# (C, T) -> (T, C)
train_mel  = [to_time_major(m) for m in train_mel]
test_mel   = [to_time_major(m) for m in test_mel]

train_mfcc = [to_time_major(m) for m in train_mfcc]
test_mfcc  = [to_time_major(m) for m in test_mfcc]

mean, std = fit_zscore(train_mfcc)

train_mfcc = apply_zscore(train_mfcc, mean, std)
test_mfcc  = apply_zscore(test_mfcc,  mean, std)

train_mel  = torch.tensor(train_mel,  dtype=torch.float32)
train_mfcc = torch.tensor(train_mfcc, dtype=torch.float32)
test_mel   = torch.tensor(test_mel,   dtype=torch.float32)
test_mfcc  = torch.tensor(test_mfcc,  dtype=torch.float32)

lr = 1e-4
epochs = 30
features_set = "mfcc"

if features_set == "mel":
    train_features = train_mel
    test_features = test_mel
elif features_set == "mfcc":
    train_features = train_mfcc
    test_features = test_mfcc
elif features_set == "combine":
    train_features = torch.cat([train_mel, train_mfcc], dim=2)
    test_features  = torch.cat([test_mel,  test_mfcc],  dim=2)
else:
    raise ValueError("features_set inválido")

input_size = train_features.shape[2] 

train_dataset = MachineAudioDataset(train_features, y_train)
test_dataset  = MachineAudioDataset(test_features,  y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

model = GRUModel(input_size).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lr)

c:\Users\josel\OneDrive\Desktop\NCIA\.venv\Lib\site-packages\librosa\feature\spectral.py:2148: UserWarning: Empty filters detected in mel frequency basis. Some channels will produce empty responses. Try increasing your sampling rate (and fmax) or reducing n_mels.
  mel_basis = filters.mel(sr=sr, n_fft=n_fft, **kwargs)
C:\Users\josel\AppData\Local\Temp\ipykernel_16564\1061274810.py:19: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  train_mel  = torch.tensor(train_mel,  dtype=torch.float32)


In [14]:
train(model, optimizer, criterion, train_loader, epochs)

C:\Users\josel\AppData\Local\Temp\ipykernel_16564\3065367794.py:13: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return (torch.tensor(feature, dtype=torch.float32),


Epoch 1/30, Loss: 0.6288
Epoch 2/30, Loss: 0.3433
Epoch 3/30, Loss: 0.2980
Epoch 4/30, Loss: 0.2860
Epoch 5/30, Loss: 0.2760
Epoch 6/30, Loss: 0.2725
Epoch 7/30, Loss: 0.2663
Epoch 8/30, Loss: 0.2569
Epoch 9/30, Loss: 0.2532
Epoch 10/30, Loss: 0.2461
Epoch 11/30, Loss: 0.2436
Epoch 12/30, Loss: 0.2376
Epoch 13/30, Loss: 0.2360
Epoch 14/30, Loss: 0.2320
Epoch 15/30, Loss: 0.2302
Epoch 16/30, Loss: 0.2324
Epoch 17/30, Loss: 0.2253
Epoch 18/30, Loss: 0.2233
Epoch 19/30, Loss: 0.2190
Epoch 20/30, Loss: 0.2193
Epoch 21/30, Loss: 0.2159
Epoch 22/30, Loss: 0.2158
Epoch 23/30, Loss: 0.2159
Epoch 24/30, Loss: 0.2067
Epoch 25/30, Loss: 0.2099
Epoch 26/30, Loss: 0.2088
Epoch 27/30, Loss: 0.2080
Epoch 28/30, Loss: 0.2091
Epoch 29/30, Loss: 0.2039
Epoch 30/30, Loss: 0.2014


In [15]:
acc, prec, rec, f1, cm = evaluate(model, test_loader)

print("=" * 80)
print(f"Acuracia: {acc:.2%}")
print(f"Precisao: {prec:.2%}")
print(f"Revocacao: {rec:.2%}")
print(f"F1: {f1:.2%}")
print(f"Matriz de confusão: {cm}")
print("=" * 80)

C:\Users\josel\AppData\Local\Temp\ipykernel_16564\3065367794.py:13: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return (torch.tensor(feature, dtype=torch.float32),


Acuracia: 94.44%
Precisao: 79.57%
Revocacao: 68.97%
F1: 72.89%
Matriz de confusão: [[661  11]
 [ 29  19]]


In [16]:
train_mel, train_mfcc = np.array(train_mel), np.array(train_mfcc)
test_mel, test_mfcc = np.array(test_mel), np.array(test_mfcc)
 
features_set = 'mfcc'

if features_set == 'mel':
    train_features = train_mel
    test_features = test_mel
elif features_set == 'mfcc':
    train_features = train_mfcc
    test_features = test_mfcc
elif features_set == 'combine':
    train_features = np.concatenate([train_mel, train_mfcc], axis=-1)
    test_features = np.concatenate([test_mel, test_mfcc], axis=-1)

train_features = np.mean(train_features, axis=1)
test_features = np.mean(test_features, axis=1)

input_size = train_features[0].shape[0]

model = DecisionTreeModel(random_state=42)
model.fit(train_features, y_train)

y_pred = model.predict(test_features)

print(f"Acuracia: {accuracy_score(y_test, y_pred):.2%}")
print(f"Precisao: {precision_score(y_test, y_pred):.2%}")
print(f"Revocacao: {recall_score(y_test, y_pred):.2%}")
print(f"F1: {f1_score(y_test, y_pred):.2%}")
#print(f"Matriz de confusao: {confusion_matrix(y_test, y_pred):.2%}")

C:\Users\josel\AppData\Local\Temp\ipykernel_16564\3016041511.py:1: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  train_mel, train_mfcc = np.array(train_mel), np.array(train_mfcc)


Acuracia: 90.56%
Precisao: 32.14%
Revocacao: 37.50%
F1: 34.62%


C:\Users\josel\AppData\Local\Temp\ipykernel_16564\3016041511.py:2: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  test_mel, test_mfcc = np.array(test_mel), np.array(test_mfcc)
